# RSM Reciprocal-Coordinate Workflow

Public-safe Jupyter notebook distilled from the GaSb/GaAs(111)A reciprocal-space-map analysis workflow. The original workflow converted measured reciprocal-space-map tables into qx/qz plots, log-intensity views, and relaxation-guide overlays for structural interpretation. Raw spreadsheets and private instrument exports are intentionally excluded from this public notebook.

## Expected Input

A cleaned table with three columns:

- `qx`: in-plane reciprocal coordinate
- `qz`: out-of-plane reciprocal coordinate
- `intensity`: measured intensity

The demonstration below uses synthetic data only so the notebook can render publicly without exposing raw measurement files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 130, "font.size": 10})

In [ ]:
def make_synthetic_rsm(nx=180, nz=220):
    """Build a toy RSM-like table for public rendering."""
    qx = np.linspace(-0.08, 0.08, nx)
    qz = np.linspace(6.45, 7.20, nz)
    qx_grid, qz_grid = np.meshgrid(qx, qz)

    substrate = np.exp(-((qx_grid / 0.010) ** 2 + ((qz_grid - 6.96) / 0.018) ** 2))
    film = 0.42 * np.exp(-(((qx_grid - 0.018) / 0.015) ** 2 + ((qz_grid - 6.78) / 0.035) ** 2))
    fringes = 0.05 * (1 + np.cos((qz_grid - 6.70) * 95)) * np.exp(-(qx_grid / 0.045) ** 2)
    intensity = 1 + 2.5e4 * (substrate + film + fringes)

    return pd.DataFrame({
        "qx": qx_grid.ravel(),
        "qz": qz_grid.ravel(),
        "intensity": intensity.ravel(),
    })

rsm = make_synthetic_rsm()
rsm.head()

In [ ]:
def plot_rsm_log(table, title="RSM log-intensity map"):
    pivot = table.pivot(index="qz", columns="qx", values="intensity")
    qx = pivot.columns.to_numpy(dtype=float)
    qz = pivot.index.to_numpy(dtype=float)
    log_intensity = np.log10(np.clip(pivot.to_numpy(), 1, None))

    fig, ax = plt.subplots(figsize=(5.8, 4.8))
    mesh = ax.pcolormesh(qx, qz, log_intensity, shading="auto", cmap="viridis")
    ax.set_xlabel(r"$q_x$ (arb. reciprocal units)")
    ax.set_ylabel(r"$q_z$ (arb. reciprocal units)")
    ax.set_title(title)
    fig.colorbar(mesh, ax=ax, label=r"$\log_{10}(I)$")
    return fig, ax

fig, ax = plot_rsm_log(rsm, title="Public-safe synthetic RSM example")

In [ ]:
def add_relaxation_guides(ax, substrate_qx=0.0, substrate_qz=6.96):
    """Add simple visual guide lines used during relaxation/strain interpretation."""
    ax.axvline(substrate_qx, color="white", lw=1.2, ls="--", label="substrate in-plane reference")
    ax.axhline(substrate_qz, color="white", lw=1.2, ls=":", label="substrate out-of-plane reference")
    ax.plot([substrate_qx, 0.035], [substrate_qz, 6.70], color="tomato", lw=1.5, label="relaxation guide")
    ax.legend(loc="lower left", fontsize=8, frameon=True)

fig, ax = plot_rsm_log(rsm, title="RSM with reciprocal-coordinate and relaxation guides")
add_relaxation_guides(ax)

## Provenance Note

This public notebook is a cleaned derivative of earlier Boise State RSM notebooks. It is intended to show the analysis pattern: read reciprocal-coordinate intensity data, render linear/log views, and overlay structural guide lines. The original raw measurement tables are not included.